# 3. Plot — 作图

1. **AUROC1**：读取 `2.a.benchmark` 的灵敏度表，绘制 Family / Superfamily / Fold，并刷新 `auc_easy.csv`。
2. **Translation accuracy**：读取 `2.b.translation_eval` 的 `translation_summary.csv`，绘制 aa2di / di2aa 柱状图。

共享配置：[`config.py`](config.py)（`METHODS` / `PALETTE` / `TRANSLATION_METRICS_DIR`）。


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

print("python:", sys.executable)

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except ModuleNotFoundError as e:
    env_py = "/hpcfs/fhome/caihuize/.conda/envs/ESM3_3Di_5090/bin/python"
    raise SystemExit(
        f"当前 kernel 缺少依赖 ({e})。不要在 miniforge3 python3.12 上 pip。\n"
        f"请换 kernel 到 ESM3_3Di_5090，或在终端用该环境出图：\n"
        f"  cd /hpcfs/fhome/caihuize/scope40_easy\n"
        f"  {env_py} -c \"import matplotlib; print(matplotlib.__version__)\"\n"
    ) from e

from IPython.display import Image, display

from config import (
    FIGURES_DIR,
    METRICS_DIR,
    METHODS,
    PALETTE,
    TRANSLATION_METRICS_DIR,
    PROJECT_ROOT,
    cleanup_tmp,
    ensure_work_dirs,
    metric_prefix,
    require_project_root,
)

ROOT = require_project_root("3.plot.ipynb")
assert ROOT == PROJECT_ROOT

ensure_work_dirs()
print("ROOT:", ROOT)
print("metrics:", METRICS_DIR)
print("figures:", FIGURES_DIR)


python: /hpcfs/fpublic/app/miniforge3/conda/bin/python3.12
缺少 matplotlib，正在从清华镜像安装 matplotlib …
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


ERROR: Could not find a version that satisfies the requirement matplotlib (from versions: none)
ERROR: No matching distribution found for matplotlib


CalledProcessError: Command '['/hpcfs/fpublic/app/miniforge3/conda/bin/python3.12', '-m', 'pip', 'install', '--user', '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple', 'matplotlib']' returned non-zero exit status 1.

In [ ]:
def roc_plot(ax, file: Path, tool: str, color: str | None = None) -> float:
    data = []
    with file.open() as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                data.append(parts)
    if not data:
        return 0.0
    data.sort(key=lambda x: float(x[3]), reverse=True)
    x = [(i + 1) / len(data) for i in range(len(data))]
    y = [float(row[3]) for row in data]
    auc_val = sum(y) / len(y)
    ax.plot(x, y, label=f"{tool} AUC={auc_val:.3f}", color=color, linewidth=1.4)
    return auc_val


def plot_auroc1_easy(
    output_png: Path | None = None,
    auc_csv: Path | None = None,
) -> tuple[Path, Path]:
    output_png = Path(output_png or (FIGURES_DIR / "auroc1_easy.png"))
    auc_csv = Path(auc_csv or (METRICS_DIR / "auc_easy.csv"))

    level_files = {"Family": "fam", "Superfamily": "sup", "Fold": "fol"}
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    auc_rows: list[dict] = []

    for ax, (title, level_key) in zip(axs, level_files.items()):
        ax.set_title(title, fontsize=16)
        row: dict[str, float | str] = {"search_mode": "easy", "level": title}
        for label, key, _engine, _di in METHODS:
            path = Path(str(metric_prefix(key)) + f"_{level_key}.tsv")
            if not path.is_file():
                print(f"❌ {label} {title}: 缺少 {path}")
                continue
            auc_val = roc_plot(ax, path, label, color=PALETTE.get(label))
            row[label] = auc_val
            print(f"✅ {label} {title}: AUC={auc_val:.4f}")
        auc_rows.append(row)
        ax.set_xlim(-0.01, 1.01)
        ax.set_ylim(-0.01, 1.01)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        legend_loc = "upper right" if title == "Fold" else "lower left"
        ax.legend(fontsize=8, loc=legend_loc)

    axs[0].set_ylabel("Fraction of TPs up to first FP", fontsize=12)
    axs[1].set_xlabel("Fraction of queries", fontsize=12)

    output_png.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(output_png, dpi=300, facecolor="white", bbox_inches="tight")
    plt.close()
    print(f"图片: {output_png}")

    df = pd.DataFrame(auc_rows)
    df.to_csv(auc_csv, index=False)
    print(f"AUC CSV: {auc_csv}")
    return output_png, auc_csv


png_path, csv_path = plot_auroc1_easy()
auc_df = pd.read_csv(csv_path)
display(auc_df)
display(Image(filename=str(png_path)))
print(f"\n图: {png_path}")
print(f"CSV: {csv_path}")


In [ ]:
def plot_translation_accuracy(
    df: pd.DataFrame,
    output_png: Path | None = None,
) -> Path | None:
    if df.empty:
        print("Translation: 无数据，跳过绘图（先运行 2.b.translation_eval.ipynb）")
        return None
    output_png = Path(output_png or (FIGURES_DIR / "translation_accuracy.png"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, task, title in zip(
        axes,
        ("aa2di", "di2aa"),
        ("AA→3Di vs GT 3Di", "3Di→AA vs GT AA"),
        strict=True,
    ):
        sub = df[df["task"] == task].sort_values("micro_acc", ascending=False)
        if sub.empty:
            ax.set_title(f"{title} (no data)")
            continue
        x = range(len(sub))
        colors = [PALETTE.get(lbl, "#888888") for lbl in sub["label"]]
        ax.bar(x, sub["micro_acc"], color=colors, alpha=0.85, label="micro")
        ax.plot(x, sub["macro_acc"], "ko-", markersize=6, label="macro")
        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["label"], rotation=25, ha="right")
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("Accuracy")
        ax.set_title(title)
        ax.legend(loc="lower right")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    fig.tight_layout()
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Translation 图: {output_png}")
    return output_png


trans_csv = TRANSLATION_METRICS_DIR / "translation_summary.csv"
if trans_csv.is_file():
    trans_df = pd.read_csv(trans_csv)
else:
    trans_df = pd.DataFrame()
    print(f"⚠️  缺少 {trans_csv}，跳过 translation 图")

trans_png = plot_translation_accuracy(trans_df)
if trans_png is not None:
    display(trans_df)
    display(Image(filename=str(trans_png)))


## 与 new_scope40 参考值对照

| Method | Family | Superfamily | Fold |
|--------|--------|-------------|------|
| Foldseek (AA+3Di) | 0.735 | 0.631 | 0.081 |
| MMseqs2 | — | — | — |
| ProstT5 (translate) | 0.708 | 0.589 | 0.055 |
| ESM3-LoRA | 0.698 | 0.582 | 0.051 |
| ESM3-3Di | 0.698 | 0.582 | 0.055 |
| SaProt | 0.458 | 0.343 | 0.024 |


## 清理临时目录

删除项目根 `tmp/` 与 `work/tmp/`（下载/解压/搜索中间文件）。产物在 `work/` 与 `bin/` 中保留。


In [ ]:
cleanup_tmp(also_work_tmp=True)
